# Step 5 - Model training and evaluation (Google Colab)

**Before you run**
1. `Runtime > Change runtime type > T4 GPU` (the CNN/BiLSTM/BERT cells are much faster on GPU).
2. Run the cells from top to bottom. When asked, upload these 4 files from `D:\ICT\SEM 7\nlppro\data\processed`:
   `features.npz`, `embedding_matrix.npy`, `vocab.json`, `pairs_features.csv`.
3. At the end a zip with all tables, figures and models downloads automatically.

**Models compared (same split, same labels):** VADER rule baseline, Naive Bayes, linear SVM, Logistic Regression, CNN,
BiLSTM (no aspect), BiLSTM + aspect attention, and optionally BERT. Main metric: **macro-F1** (the data is imbalanced).

In [ ]:
# ===== 0. Setup =====
import subprocess, sys
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'vaderSentiment'], check=False)

import os, json, random, copy, re, time, warnings
import numpy as np, pandas as pd
import torch, torch.nn as nn
import matplotlib
import matplotlib.pyplot as plt
from sklearn.metrics import accuracy_score, f1_score, classification_report, ConfusionMatrixDisplay
from sklearn.utils.class_weight import compute_class_weight
warnings.filterwarnings('ignore')

IN_COLAB = 'google.colab' in sys.modules
DATA = os.environ.get('ASC_DATA', '/content/data')
OUT = os.environ.get('ASC_OUT', '/content/outputs')
for d in (DATA, OUT + '/figures', OUT + '/tables', OUT + '/models'):
    os.makedirs(d, exist_ok=True)

if IN_COLAB and not os.path.exists(f'{DATA}/features.npz'):
    from google.colab import files
    print('Select the 4 files: features.npz, embedding_matrix.npy, vocab.json, pairs_features.csv')
    for name, content in files.upload().items():
        open(f'{DATA}/{name}', 'wb').write(content)

dev = 'cuda' if torch.cuda.is_available() else 'cpu'
print('device:', dev, '| GPU:', torch.cuda.get_device_name(0) if dev == 'cuda' else 'none (switch the runtime to GPU for speed)')

In [ ]:
# ===== 1. Load the Step 4 files =====
Z = np.load(f'{DATA}/features.npz', allow_pickle=True)     # the 'split' array was saved as text objects
A = {k: Z[k] for k in Z.files}
EMB = np.load(f'{DATA}/embedding_matrix.npy')
VOCAB = json.load(open(f'{DATA}/vocab.json'))
INV = {i: w for w, i in VOCAB.items()}
df = pd.read_csv(f'{DATA}/pairs_features.csv', dtype={'sentence_id': str})
for c in ('ctx', 'ctx_window', 'tokens'):
    df[c] = df[c].fillna('')
assert len(df) == len(A['y']), 'features.npz and pairs_features.csv must have the same number of rows'

LABELS = ['negative', 'neutral', 'positive']                  # ids 0, 1, 2
y, split = A['y'], A['split'].astype(str)
tr, va, te = split == 'train', split == 'val', split == 'test'
print('rows  train/val/test:', tr.sum(), va.sum(), te.sum(), '| embedding matrix:', EMB.shape)
CLASS_W = compute_class_weight('balanced', classes=np.array([0, 1, 2]), y=y[tr])
print('class weights (train):', dict(zip(LABELS, CLASS_W.round(3).tolist())))

def metrics(yt, yp):
    return dict(acc=accuracy_score(yt, yp), macro_f1=f1_score(yt, yp, average='macro'),
                weighted_f1=f1_score(yt, yp, average='weighted'))

RESULTS = {}      # model name -> dict(val_f1, test=metrics, std, cv, cv_std, pred=test predictions)

In [ ]:
# ===== 2. Rule-based baseline: VADER on the raw aspect window (+-5 words) =====
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer
sia = SentimentIntensityAnalyzer()

def raw_window(text, s, e, k=5):
    spans = [(m.start(), m.end()) for m in re.finditer(r'\S+', text)]
    idx = [i for i, (a, b) in enumerate(spans) if a < e and b > s]
    if not idx: return text
    lo, hi = max(idx[0] - k, 0), min(idx[-1] + k + 1, len(spans))
    return text[spans[lo][0]:spans[hi - 1][1]]

def vader_label(t):
    c = sia.polarity_scores(t)['compound']
    return 2 if c >= 0.05 else 0 if c <= -0.05 else 1

df['raw_window'] = [raw_window(t, s, e) for t, s, e in zip(df.text, df.start, df.end)]
vp = np.array([vader_label(t) for t in df.raw_window])
RESULTS['Rule-based (VADER)'] = dict(val_f1=f1_score(y[va], vp[va], average='macro'), test=metrics(y[te], vp[te]), pred=vp[te])
print('VADER  val macro-F1 %.3f | test macro-F1 %.3f' % (RESULTS['Rule-based (VADER)']['val_f1'], RESULTS['Rule-based (VADER)']['test']['macro_f1']))

In [ ]:
# ===== 3. Classical ML on TF-IDF (small grid chosen on VALIDATION, then 5-fold grouped CV) =====
from sklearn.pipeline import make_pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.naive_bayes import MultinomialNB
from sklearn.model_selection import GroupKFold, cross_val_score

def tfidf_pipe(clf):
    return make_pipeline(TfidfVectorizer(ngram_range=(1, 3), min_df=2, sublinear_tf=True), clf)

GRIDS = {
    'Naive Bayes': [(MultinomialNB, dict(alpha=a)) for a in (0.1, 0.5, 1.0)],
    'SVM (linear)': [(LinearSVC, dict(C=c, class_weight='balanced')) for c in (0.05, 0.1, 0.5, 1.0)],
    'Logistic Regression': [(LogisticRegression, dict(C=c, class_weight='balanced', max_iter=2000)) for c in (0.5, 1, 5, 10)],
}
trva = tr | va
groups = df.sentence_id.values[trva]
BEST_PIPE = {}

def fit_classical(name, grid, col):
    best = None
    for cls, params in grid:
        p = tfidf_pipe(cls(**params)).fit(df[col][tr], y[tr])
        f = f1_score(y[va], p.predict(df[col][va]), average='macro')
        if best is None or f > best[0]: best = (f, cls, params, p)
    f, cls, params, p = best
    cv = cross_val_score(tfidf_pipe(cls(**params)), df[col][trva], y[trva], groups=groups,
                         cv=GroupKFold(5), scoring='f1_macro')
    pred = p.predict(df[col][te])
    return dict(val_f1=f, test=metrics(y[te], pred), cv=cv.mean(), cv_std=cv.std(), pred=pred, params=params), p

for name, grid in GRIDS.items():
    RESULTS[name], BEST_PIPE[name] = fit_classical(name, grid, 'ctx')
    r = RESULTS[name]
    print(f"{name:<22} val F1 {r['val_f1']:.3f} | test F1 {r['test']['macro_f1']:.3f} | 5-fold CV {r['cv']:.3f} +- {r['cv_std']:.3f} | {r['params']}")

# comparison row: same Logistic Regression but on the WHOLE sentence (ignores which aspect is asked about)
RESULTS['LogReg - whole sentence (no aspect)'], _ = fit_classical('LR whole', GRIDS['Logistic Regression'], 'tokens')
r = RESULTS['LogReg - whole sentence (no aspect)']
print(f"{'LR whole sentence':<22} val F1 {r['val_f1']:.3f} | test F1 {r['test']['macro_f1']:.3f} | 5-fold CV {r['cv']:.3f} +- {r['cv_std']:.3f}")

In [ ]:
# ===== 4. Neural models =====
import torch.nn.functional as F
from torch.nn.utils.rnn import pack_padded_sequence, pad_packed_sequence

T = dict(sent=torch.tensor(A['sent_ids']), asp=torch.tensor(A['asp_ids']), ctx=torch.tensor(A['ctx_ids']),
         len=torch.tensor(A['sent_len']), y=torch.tensor(y))
KEYS = ['sent', 'asp', 'ctx', 'len', 'y']

def make_loader(mask, bs, shuffle):
    m = torch.from_numpy(mask)
    return torch.utils.data.DataLoader(torch.utils.data.TensorDataset(*[T[k][m] for k in KEYS]), batch_size=bs, shuffle=shuffle)

def emb_layer():
    return nn.Embedding.from_pretrained(torch.tensor(EMB), freeze=False, padding_idx=0)

class CNNClassifier(nn.Module):
    # CNN over the aspect-context window (+-5 words): 3 filter widths, max-pooling over time
    def __init__(self, n_filters=100, ks=(3, 4, 5), drop=0.5):
        super().__init__()
        self.emb = emb_layer(); d = EMB.shape[1]
        self.convs = nn.ModuleList([nn.Conv1d(d, n_filters, k, padding=k // 2) for k in ks])
        self.drop, self.fc = nn.Dropout(drop), nn.Linear(n_filters * len(ks), 3)
    def forward(self, sent, asp, ctx, length):
        x = self.emb(ctx).transpose(1, 2)
        h = torch.cat([torch.relu(c(x)).max(dim=2).values for c in self.convs], 1)
        return self.fc(self.drop(h)), None

class BiLSTMAttn(nn.Module):
    # BiLSTM over the whole sentence + attention. use_aspect=True: the aspect vector is fed to every word AND to the
    # attention scorer, so the model learns which words matter for THIS aspect. use_aspect=False is the ablation.
    def __init__(self, use_aspect=True, hid=128, drop=0.5):
        super().__init__()
        self.use_aspect = use_aspect; d = EMB.shape[1]
        self.emb = emb_layer()
        self.lstm = nn.LSTM(d * (2 if use_aspect else 1), hid, batch_first=True, bidirectional=True)
        self.att = nn.Linear(2 * hid + (d if use_aspect else 0), 1)
        self.drop, self.fc = nn.Dropout(drop), nn.Linear(2 * hid, 3)
    def forward(self, sent, asp, ctx, length):
        x, mask = self.emb(sent), sent != 0
        a = None
        if self.use_aspect:
            m = (asp != 0).float().unsqueeze(-1)
            a = ((self.emb(asp) * m).sum(1) / m.sum(1).clamp(min=1)).unsqueeze(1).expand(-1, x.size(1), -1)
            x = torch.cat([x, a], -1)
        h, _ = self.lstm(pack_padded_sequence(self.drop(x), length.cpu(), batch_first=True, enforce_sorted=False))
        h, _ = pad_packed_sequence(h, batch_first=True, total_length=sent.size(1))
        score = self.att(torch.cat([h, a], -1) if a is not None else h).squeeze(-1).masked_fill(~mask, -1e9)
        w = torch.softmax(score, dim=1).unsqueeze(-1)
        return self.fc(self.drop((w * h).sum(1))), w.squeeze(-1)

@torch.no_grad()
def predict(model, mask, with_att=False):
    model.eval(); preds, atts = [], []
    for sent, asp, ctx, ln, yb in make_loader(mask, 256, False):
        logits, w = model(sent.to(dev), asp.to(dev), ctx.to(dev), ln)
        preds.append(logits.argmax(1).cpu())
        if with_att and w is not None: atts.append(w.cpu())
    return torch.cat(preds).numpy(), (torch.cat(atts).numpy() if atts else None)

loss_fn = nn.CrossEntropyLoss(weight=torch.tensor(CLASS_W, dtype=torch.float).to(dev))

def train_one(make_model, seed, max_epochs=40, patience=6, lr=1e-3, bs=32):
    torch.manual_seed(seed); np.random.seed(seed); random.seed(seed)
    model = make_model().to(dev)
    opt = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=1e-5)
    best_f1, best_state, best_ep, bad = -1, None, 0, 0
    for ep in range(1, max_epochs + 1):
        model.train()
        for sent, asp, ctx, ln, yb in make_loader(tr, bs, True):
            loss = loss_fn(model(sent.to(dev), asp.to(dev), ctx.to(dev), ln)[0], yb.to(dev))
            opt.zero_grad(); loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), 5.0); opt.step()
        f1 = f1_score(y[va], predict(model, va)[0], average='macro')
        if f1 > best_f1: best_f1, best_state, best_ep, bad = f1, copy.deepcopy(model.state_dict()), ep, 0
        else: bad += 1
        if bad >= patience: break
    model.load_state_dict(best_state)
    return model, best_f1, best_ep

SEEDS = [42, 43, 44]
DEEP = {'CNN (aspect window)': lambda: CNNClassifier(),
        'BiLSTM (no aspect)': lambda: BiLSTMAttn(use_aspect=False),
        'BiLSTM + aspect attention': lambda: BiLSTMAttn(use_aspect=True)}
BEST_MODEL = {}

for name, maker in DEEP.items():
    runs = []
    for seed in SEEDS:
        t0 = time.time()
        model, vf, ep = train_one(maker, seed)
        pred = predict(model, te)[0]
        runs.append(dict(val_f1=vf, pred=pred, m=metrics(y[te], pred), model=model))
        print(f"{name:<27} seed {seed} | best epoch {ep:2d} | val F1 {vf:.3f} | test F1 {runs[-1]['m']['macro_f1']:.3f} | {time.time() - t0:.0f}s")
    top = max(runs, key=lambda r: r['val_f1'])                      # representative run = best on validation
    BEST_MODEL[name] = top['model']
    mean = lambda k: float(np.mean([r['m'][k] for r in runs]))
    RESULTS[name] = dict(val_f1=float(np.mean([r['val_f1'] for r in runs])),
                         test=dict(acc=mean('acc'), macro_f1=mean('macro_f1'), weighted_f1=mean('weighted_f1')),
                         std=float(np.std([r['m']['macro_f1'] for r in runs])), pred=top['pred'])

In [ ]:
# ===== 5. Comparison table =====
base = RESULTS['Rule-based (VADER)']['test']['macro_f1']
rows = []
for name, r in RESULTS.items():
    rows.append(dict(model=name, val_macro_f1=r['val_f1'], test_acc=r['test']['acc'], test_macro_f1=r['test']['macro_f1'],
                     test_macro_f1_std=r.get('std', np.nan), test_weighted_f1=r['test']['weighted_f1'],
                     cv5_macro_f1=r.get('cv', np.nan), cv5_std=r.get('cv_std', np.nan),
                     pct_over_rule_baseline=(r['test']['macro_f1'] - base) / base * 100))
table = pd.DataFrame(rows).round(3)
table.to_csv(f'{OUT}/tables/model_comparison.csv', index=False)
pd.set_option('display.width', 220); pd.set_option('display.max_columns', 20)
print(table.to_string(index=False))
print('\nDeep models: test numbers are the MEAN over 3 seeds (std in test_macro_f1_std). Classical: single run + 5-fold grouped CV.')

In [ ]:
# ===== 6. Confusion matrices, classification report, attention examples =====
classical = [n for n in RESULTS if n in GRIDS]
best_cl = max(classical, key=lambda n: RESULTS[n]['val_f1'])
best_dl = max(DEEP, key=lambda n: RESULTS[n]['val_f1'])
print('best classical (by val):', best_cl, '| best deep (by val):', best_dl)

for tag, name in [('classical', best_cl), ('deep', best_dl)]:
    disp = ConfusionMatrixDisplay.from_predictions(y[te], RESULTS[name]['pred'], display_labels=LABELS, cmap='Blues')
    disp.ax_.set_title(f'{name} - test set')
    plt.savefig(f"{OUT}/figures/cm_{tag}.png", dpi=200, bbox_inches='tight'); plt.show()

print(f'\nClassification report - {best_dl} (test, best-validation seed):')
print(classification_report(y[te], RESULTS[best_dl]['pred'], target_names=LABELS, digits=3))

# what does the aspect attention look at?
att_name = 'BiLSTM + aspect attention'
pred, att = predict(BEST_MODEL[att_name], te, with_att=True)
dte = df[te].reset_index(drop=True)
sent_te, len_te = A['sent_ids'][te], A['sent_len'][te]
print('\nAttention examples (top-3 words the model looked at for the given aspect):')
for i in range(6):
    n = len_te[i]; top3 = att[i][:n].argsort()[::-1][:3]
    print(f"aspect={dte.term[i]!r:<22} gold={LABELS[y[te][i]]:<9} pred={LABELS[pred[i]]:<9} looked at: {[INV[sent_te[i][j]] for j in top3]}")

# save predictions for the Step 6 error analysis
out = dte[['sentence_id', 'text', 'term', 'polarity']].copy()
for name in RESULTS: out['pred_' + name] = [LABELS[p] for p in RESULTS[name]['pred']]
out.to_csv(f'{OUT}/tables/test_predictions.csv', index=False)

In [ ]:
# ===== 7. Save models, zip everything, download =====
import joblib, shutil
torch.save(dict(model=att_name, state=BEST_MODEL[att_name].state_dict(), vocab=VOCAB, emb_shape=EMB.shape), f'{OUT}/models/asc_bilstm_attn.pt')
torch.save(dict(model=best_dl, state=BEST_MODEL[best_dl].state_dict(), vocab=VOCAB, emb_shape=EMB.shape), f'{OUT}/models/asc_best_deep.pt')
joblib.dump(BEST_PIPE[best_cl], f'{OUT}/models/asc_{best_cl.split()[0].lower()}_tfidf.joblib')
shutil.make_archive(os.path.dirname(OUT) + '/step5_outputs', 'zip', OUT)
print('saved ->', os.listdir(OUT + '/tables'), os.listdir(OUT + '/figures'), os.listdir(OUT + '/models'))
if IN_COLAB:
    from google.colab import files
    files.download('/content/step5_outputs.zip')

## Optional: BERT (needs the GPU runtime)
The sentence and the aspect are given to BERT as a pair: `[CLS] sentence [SEP] aspect [SEP]`. This is usually the strongest model
on this task. It adds about 3-5 minutes on a T4. Skip it if you are short on time - everything above is already saved.

In [ ]:
# ===== 8. (optional) BERT sentence-pair classifier =====
RUN_BERT = True
if RUN_BERT and dev == 'cuda':
    from transformers import AutoTokenizer, AutoModelForSequenceClassification, get_linear_schedule_with_warmup
    torch.manual_seed(42); np.random.seed(42); random.seed(42)
    tok = AutoTokenizer.from_pretrained('bert-base-uncased')

    def bert_ds(mask):
        e = tok(df.text[mask].tolist(), df.term[mask].tolist(), truncation=True, padding='max_length', max_length=96, return_tensors='pt')
        return torch.utils.data.TensorDataset(e['input_ids'], e['attention_mask'], e['token_type_ids'], torch.tensor(y[mask]))

    dtr, dva, dte_b = bert_ds(tr), bert_ds(va), bert_ds(te)
    bert = AutoModelForSequenceClassification.from_pretrained('bert-base-uncased', num_labels=3).to(dev)
    EPOCHS = 4
    tl = torch.utils.data.DataLoader(dtr, batch_size=16, shuffle=True)
    opt = torch.optim.AdamW(bert.parameters(), lr=2e-5, weight_decay=0.01)
    sch = get_linear_schedule_with_warmup(opt, int(0.1 * EPOCHS * len(tl)), EPOCHS * len(tl))
    bert_loss = nn.CrossEntropyLoss(weight=torch.tensor(CLASS_W, dtype=torch.float).to(dev))

    @torch.no_grad()
    def bert_predict(ds):
        bert.eval(); out = []
        for ids, am, tt, yb in torch.utils.data.DataLoader(ds, batch_size=64):
            out.append(bert(input_ids=ids.to(dev), attention_mask=am.to(dev), token_type_ids=tt.to(dev)).logits.argmax(1).cpu())
        return torch.cat(out).numpy()

    best_f1, best_state = -1, None
    for ep in range(1, EPOCHS + 1):
        bert.train()
        for ids, am, tt, yb in tl:
            logits = bert(input_ids=ids.to(dev), attention_mask=am.to(dev), token_type_ids=tt.to(dev)).logits
            loss = bert_loss(logits, yb.to(dev))
            opt.zero_grad(); loss.backward(); nn.utils.clip_grad_norm_(bert.parameters(), 1.0); opt.step(); sch.step()
        f1 = f1_score(y[va], bert_predict(dva), average='macro')
        print(f'BERT epoch {ep} | val macro-F1 {f1:.3f}')
        if f1 > best_f1: best_f1, best_state = f1, copy.deepcopy(bert.state_dict())
    bert.load_state_dict(best_state)
    bp = bert_predict(dte_b)
    RESULTS['BERT (sentence + aspect)'] = dict(val_f1=best_f1, test=metrics(y[te], bp), pred=bp)
    r = RESULTS['BERT (sentence + aspect)']['test']
    print(f"BERT test: accuracy {r['acc']:.3f} | macro-F1 {r['macro_f1']:.3f} | weighted-F1 {r['weighted_f1']:.3f}")
    table.loc[len(table)] = dict(model='BERT (sentence + aspect)', val_macro_f1=round(best_f1, 3), test_acc=round(r['acc'], 3),
                                 test_macro_f1=round(r['macro_f1'], 3), test_macro_f1_std=np.nan, test_weighted_f1=round(r['weighted_f1'], 3),
                                 cv5_macro_f1=np.nan, cv5_std=np.nan, pct_over_rule_baseline=round((r['macro_f1'] - base) / base * 100, 3))
    table.to_csv(f'{OUT}/tables/model_comparison.csv', index=False)
    print(table.to_string(index=False))
    if IN_COLAB:
        shutil.make_archive(os.path.dirname(OUT) + '/step5_outputs', 'zip', OUT)
        files.download('/content/step5_outputs.zip')
else:
    print('BERT skipped (set RUN_BERT = True and use a GPU runtime to run it).')